In [1]:
import cv2
import numpy as np
import mediapipe as mp
import joblib
from ultralytics import YOLO
import pyttsx3

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

def normalize_landmarks(landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])
    coords -= coords[0].copy()
    scale = np.linalg.norm(coords[12])
    if scale > 0:
        coords /= scale
    return coords.flatten()

best_model = joblib.load('best_model.pkl')  # your letters Random Forest
model_yolo = YOLO('runs/classify/train/weights/best.pt')  # your trained YOLO digits model

I0000 00:00:1785307168.329809 1354427 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1785307168.336773 1354566 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785307168.343802 1354565 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [2]:
CONFIDENCE_MARGIN = 0.15  # digit model must beat letter model by at least this much

def get_prediction(frame, hand_landmarks_row):
    # Letter prediction (landmarks-based)
    letter_probs = best_model.predict_proba([hand_landmarks_row])[0]
    letter_pred = best_model.classes_[np.argmax(letter_probs)]
    letter_conf = np.max(letter_probs)

    # Digit prediction (image-based, YOLO)
    results = model_yolo(frame, verbose=False)
    digit_conf = results[0].probs.top1conf.item()
    digit_pred = model_yolo.names[results[0].probs.top1]

    if digit_conf > (letter_conf + CONFIDENCE_MARGIN):
        return digit_pred, 'digit', digit_conf
    else:
        return letter_pred, 'letter', letter_conf